# Experiment E7: QLoRA Fine-Tuning for Indonesian Regulatory Question Router

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/robitalhazmi/easypasal/blob/feat/phase6-replay-ab/notebooks/e7_qlora_training.ipynb)

> **Project:** EasyPasal — Citation-Grounded RAG for Indonesian Financial Regulations (OJK & Bank Indonesia)
> **Hardware Requirement:** Free GPU (Google Colab NVIDIA Tesla T4 16GB or Kaggle P100/T4)
> **Task:** Parameter-efficient 4-bit QLoRA fine-tuning of `Qwen/Qwen2.5-0.5B-Instruct` as a multi-task question router (`theme` + `is_answerable` + `is_injection`).

---
### 🛡️ Honest-Claims Protocol Notice (Plan §7 & §13)
This experiment fine-tunes a **narrow, lightweight question router and guardrail classifier** (<25ms latency) to fast-path unanswerable queries and prompt injections before expensive retrieval. It is **not** a general-purpose model fine-tuned on commercial legal drafting.

## 1. Verify GPU Hardware Acceleration

In [ ]:
!nvidia-smi
import torch

print("PyTorch CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print(
        "Device Memory:",
        round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2),
        "GB",
    )

## 2. Install Dependencies

In [ ]:
!pip install -q peft trl bitsandbytes transformers accelerate datasets pyyaml pydantic

## 3. Clone Repository & Setup Environment

In [ ]:
import os
import sys

if not os.path.exists("easypasal"):
    !git clone -b feat/phase6-replay-ab https://github.com/robitalhazmi/easypasal.git
    %cd easypasal
else:
    %cd easypasal

sys.path.insert(0, os.path.abspath("."))
sys.path.insert(0, os.path.abspath("src"))

## 4. Prepare Router Datasets

In [ ]:
from experiments.e7_qlora.dataset import build_router_dataset

counts = build_router_dataset(gold_dir="data/gold", output_dir="data/e7_router")
print("Dataset Generation Completed:", counts)

## 5. Execute Baseline Zero-Shot Evaluation (Before Fine-Tuning)

In [ ]:
from experiments.e7_qlora.evaluate import evaluate_dataset

base_dev = evaluate_dataset("data/e7_router/dev.jsonl", mode="baseline_zero_shot")
base_test = evaluate_dataset("data/e7_router/test.jsonl", mode="baseline_zero_shot")
print(
    f"Baseline Dev Theme F1: {base_dev.theme_macro_f1:.4f} | Ans F1: {base_dev.answerability_metrics.f1:.4f} | Exact Match: {base_dev.overall_exact_match:.4f}"
)
print(
    f"Baseline Test Theme F1: {base_test.theme_macro_f1:.4f} | Ans F1: {base_test.answerability_metrics.f1:.4f} | Exact Match: {base_test.overall_exact_match:.4f}"
)

## 6. Execute QLoRA Fine-Tuning with PEFT + TRL

In [ ]:
from experiments.e7_qlora.config import QLoRAConfig
from experiments.e7_qlora.train import run_gpu_training

config = QLoRAConfig.from_yaml("experiments/e7_qlora/training_config.yaml")
print("QLoRA Configuration:", config.to_dict())

# Execute SFTTrainer on GPU
training_metrics = run_gpu_training(config)
print("Training Complete! Metrics:", training_metrics)

## 7. Evaluate Fine-Tuned Adapter & Run Category Regression Audit

In [ ]:
from experiments.e7_qlora.evaluate import print_comparison_table, run_full_comparison

report = run_full_comparison(
    dev_path="data/e7_router/dev.jsonl",
    test_path="data/e7_router/test.jsonl",
    output_file="results/e7_qlora_evaluation.json",
)
print_comparison_table(report)